Cleaning data

In [16]:
import pandas as pd
import unicodedata

# Load files (notebook is in /notebook, csv files are in workspace root)
nba = pd.read_csv("../nba_alls.csv", encoding="latin1")
salary_raw = pd.read_csv("../salary_dataset.csv", header=None, encoding="latin1")

# salary_dataset.csv layout: data starts after first 2 rows,
# player name in column 1 and salary in column 2
salary = salary_raw.iloc[2:, [1, 2]].copy()
salary.columns = ["Player", "Salary"]
salary = salary.dropna(subset=["Player"])
salary["Player"] = salary["Player"].astype(str).str.strip()
salary["Salary"] = (
    salary["Salary"]
    .astype(str)
    .str.replace(r"[^0-9.-]", "", regex=True)
    .replace("", pd.NA)
    .pipe(pd.to_numeric, errors="coerce")
)

def normalize_name(value):
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.strip().lower()

nba["name_key"] = nba["Player"].map(normalize_name)
salary["name_key"] = salary["Player"].map(normalize_name)

salary_lookup = salary[["name_key", "Salary"]].drop_duplicates(subset=["name_key"])

# Keep only NBA rows for players that exist in salary_dataset.csv
matched_nba_rows = nba.merge(salary_lookup, on="name_key", how="inner").drop(columns=["name_key"])

print("Matched rows (players existing in salary_dataset.csv):", len(matched_nba_rows))
matched_nba_rows.head(20)

Matched rows (players existing in salary_dataset.csv): 0


,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,Player-additional,Salary


In [15]:
matched_nba_rows.to_csv("tested.csv", index=False)
print("Saved to tested.csv")

Saved to tested.csv


In [11]:
import pandas as pd
import unicodedata
import os

def norm_name(x):
    s = unicodedata.normalize("NFKD", str(x))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.strip().lower()

def read_csv_fallback(path, **kwargs):
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode: {path}")

# Handle filename typo: "matcted_nba_rows.csv" vs "matched_nba_rows.csv"
matched_path = "matcted_nba_rows.csv"
if not os.path.exists(matched_path):
    matched_path = "matched_nba_rows.csv"

matched_df = read_csv_fallback(matched_path)
salary_raw = read_csv_fallback("salary_dataset.csv", header=None)

# Names from matched_nba_rows.csv -> list
matched_names = []
for name in matched_df["Player"]:
    if pd.notna(name) and str(name).strip() != "":
        matched_names.append(norm_name(name))

# Names from salary_dataset.csv -> list
# (player names are column 1, data starts at row index 2)
salary_names = []
for i in range(2, len(salary_raw)):
    name = salary_raw.iloc[i, 1]
    if pd.notna(name) and str(name).strip() != "":
        salary_names.append(norm_name(name))

# Compare
matched_set = set(matched_names)
salary_set = set(salary_names)

missing_in_matched = sorted(salary_set - matched_set)   # in salary, not in matched
extra_in_matched = sorted(matched_set - salary_set)     # in matched, not in salary

print("Unique names in matched:", len(matched_set))
print("Unique names in salary :", len(salary_set))
print("Missing in matched     :", len(missing_in_matched))
print(missing_in_matched)
print("Extra in matched       :", len(extra_in_matched))
print(extra_in_matched)

Unique names in matched: 230
Unique names in salary : 228
Missing in matched     : 11
['alperen azenga1⁄4n', 'bogdan bogdanovia‡', 'dario saric', 'dennis schra¶der', 'jonas valana\x8dia«nas', 'jusuf nurkia‡', 'kristaps porzia†a£is', 'luka doncic', 'nikola jokic', 'nikola jovic', 'nikola vua\x8devia‡']
Extra in matched       : 13
['a.j. green', 'alperen ?engun', 'bogdan bogdanovi?', 'dario ?ari?', 'dennis schroder', 'jonas valan?i?nas', 'jusuf nurki?', 'kristaps porzi??is', 'luka don?i?', 'nikola joki?', 'nikola jovi?', 'nikola vu?evi?', 'toumani camara']


In [20]:
import os
import pandas as pd

# Load regular salary data (support either root or data/ path)
salary_path = '../data/old_salary.csv'
if not os.path.exists(salary_path):
    salary_path = '../data/old_salary.csv'

regular_salary = pd.read_csv(salary_path, encoding='latin1')

if 'Pos' not in regular_salary.columns:
    raise ValueError(f"'Pos' column not found. Available columns: {list(regular_salary.columns)}")

# Normalize position text and split combined positions like 'SF-PF'
pos_series = regular_salary['Pos'].fillna('').astype(str).str.upper().str.replace(' ', '', regex=False)
pos_lists = pos_series.str.split('-')

# Create binary columns for each position token, e.g., Pos_PG, Pos_SG, Pos_SF, etc.
all_pos_tokens = sorted({p for row in pos_lists for p in row if p})
for p in all_pos_tokens:
    regular_salary[f'Pos_{p}'] = pos_lists.apply(lambda row: int(p in row))

# Save and preview
output_path = '../data/old_salary_with_pos_dummies.csv'
regular_salary.to_csv(output_path, index=False)

print(f"Loaded: {salary_path}")
print(f"Added position dummy columns: {[f'Pos_{p}' for p in all_pos_tokens]}")
print(f"Saved: {output_path}")
print("\nPreview:")
preview_cols = ['Player', 'Pos'] + [f'Pos_{p}' for p in all_pos_tokens]
print(regular_salary[preview_cols].head(15).to_string(index=False))

Loaded: ../data/old_salary.csv
Added position dummy columns: ['Pos_C', 'Pos_PF', 'Pos_PG', 'Pos_SF', 'Pos_SG']
Saved: ../data/old_salary_with_pos_dummies.csv

Preview:
                 Player Pos  Pos_C  Pos_PF  Pos_PG  Pos_SF  Pos_SG
Shai Gilgeous-Alexander  PG      0       0       1       0       0
          Stephen Curry  PG      0       0       1       0       0
           Nikola Joki?   C      1       0       0       0       0
            Joel Embiid   C      1       0       0       0       0
           Kevin Durant  PF      0       1       0       0       0
  Giannis Antetokounmpo  PF      0       1       0       0       0
           Jayson Tatum  PF      0       1       0       0       0
          Anthony Davis   C      1       0       0       0       0
           Jimmy Butler  SF      0       0       0       1       0
           Devin Booker  SG      0       0       0       0       1
     Karl-Anthony Towns   C      1       0       0       0       0
           Jaylen Brown  SF 

## Playoff advanced data clean
Salary	Regular_Player_Source	Awards	Pos_C	Pos_PF	Pos_PG	Pos_SF	Pos_SG


In [4]:
import os
import pandas as pd
import unicodedata


def normalize_name(value):
    text = unicodedata.normalize('NFKD', str(value))
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text.strip().lower()


def read_csv_fallback(path, **kwargs):
    for encoding in ['utf-8', 'cp1252', 'latin1']:
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except UnicodeDecodeError:
            continue
    raise ValueError(f'Could not decode: {path}')


salary_path = 'data/playoff_salary.csv' if os.path.exists('data/playoff_salary.csv') else 'playoff_salary.csv'
adv_path = '2025_playoff_adv_raw.csv' if os.path.exists('2025_playoff_adv_raw.csv') else 'data/2025_playoff_adv_raw.csv'

salary_df = read_csv_fallback(salary_path)
adv_df = read_csv_fallback(adv_path)

salary_cols = ['Player', 'Salary', 'Awards', 'Pos_C', 'Pos_PF', 'Pos_PG', 'Pos_SF', 'Pos_SG']
salary_lookup = salary_df[salary_cols].drop_duplicates(subset=['Player']).copy()
salary_lookup['name_key'] = salary_lookup['Player'].map(normalize_name)
adv_df['name_key'] = adv_df['Player'].map(normalize_name)

salary_names = set(salary_lookup['name_key'])
adv_names = set(adv_df['name_key'])

missing_in_salary = sorted(adv_names - salary_names)
missing_in_adv = sorted(salary_names - adv_names)

print('Players in advanced raw but not in salary file:')
print(missing_in_salary)
print('Players in salary file but not in advanced raw:')
print(missing_in_adv)

# Keep only advanced rows that exist in the salary file
adv_df = adv_df[adv_df['name_key'].isin(salary_names)].copy()

matched_df = adv_df.merge(
    salary_lookup.drop(columns=['Player']),
    on='name_key',
    how='left'
)

if 'Awards_y' in matched_df.columns:
    matched_df['Awards'] = matched_df['Awards_y']
    matched_df = matched_df.drop(columns=['Awards_y'])
if 'Awards_x' in matched_df.columns:
    matched_df = matched_df.drop(columns=['Awards_x'])

matched_df = matched_df.drop(columns=['name_key'])

print(f'Loaded salary data from: {salary_path}')
print(f'Loaded advanced data from: {adv_path}')
print(f'Matched rows: {len(matched_df)}')
matched_df.head(20)


Players in advanced raw but not in salary file:
['aaron holiday', 'ajay mitchell', 'alec burks', 'alex len', 'alperen sengun', 'andre jackson jr.', 'anthony black', 'ariel hukporti', 'ausar thompson', 'baylor scheierman', 'ben sheppard', 'ben simmons', 'bogdan bogdanovic', 'brandin podziemski', 'braxton key', 'bronny james', 'caleb houstan', 'cam christie', 'cam whitmore', 'cameron payne', 'cason wallace', 'chris livingston', 'chuma okeke', 'cole anthony', 'cory joseph', 'craig porter jr.', 'dalton knecht', 'deandre jordan', 'delon wright', 'dennis schroder', 'dillon jones', 'drew eubanks', 'gary harris', 'gary payton ii', 'gary trent jr.', 'gui santos', 'hunter tyson', 'jaime jaquez jr.', 'jalen duren', 'jalen pickett', 'james johnson', 'javonte green', 'jaxson hayes', 'jay huff', 'jaylen clark', 'jaylon tyson', 'jd davison', 'jeff green', 'jericho sims', 'jett howard', 'jock landale', 'johnny furphy', 'jordan goodwin', 'jordan miller', 'jordan walsh', 'josh minott', 'julian strawther

,Player,Age,Team,Pos,G,GS,MP,PER,TS%,3PAr,...,DBPM,BPM,VORP,Salary,Pos_C,Pos_PF,Pos_PG,Pos_SF,Pos_SG,Awards
0,Shai Gilgeous-Alexander,26.0,OKC,PG,23.0,23.0,851.0,25.3,0.574,0.224,...,2.6,8.3,2.2,61005000,0,0,1,0,0,1
1,Jalen Williams,23.0,OKC,SG,23.0,23.0,796.0,19.1,0.544,0.290,...,1.7,4.2,1.2,41500000,0,0,0,0,1,1
2,Tyrese Haliburton,24.0,IND,PG,23.0,23.0,772.0,20.0,0.581,0.505,...,0.8,4.7,1.3,45550512,0,0,1,0,0,0
3,Pascal Siakam,30.0,IND,PF,23.0,23.0,771.0,20.3,0.598,0.240,...,1.1,4.3,1.2,45550512,0,1,0,0,0,1
4,Andrew Nembhard,25.0,IND,SG,23.0,23.0,769.0,12.9,0.590,0.386,...,1.3,1.0,0.6,18797619,0,0,0,0,1,0
5,Mikal Bridges,28.0,NYK,SF,18.0,18.0,706.0,12.3,0.518,0.319,...,1.2,0.3,0.4,33482145,0,0,0,1,0,0
6,OG Anunoby,27.0,NYK,PF,18.0,18.0,705.0,13.1,0.539,0.490,...,1.2,1.0,0.5,39568966,0,1,0,0,0,0
7,Chet Holmgren,22.0,OKC,C,23.0,23.0,686.0,18.1,0.565,0.342,...,1.5,2.4,0.8,41500000,1,0,0,0,0,0
8,Jalen Brunson,28.0,NYK,PG,18.0,18.0,680.0,22.2,0.582,0.341,...,-0.8,4.9,1.2,35000000,0,0,1,0,0,1
9,Myles Turner,28.0,IND,C,23.0,23.0,675.0,14.9,0.607,0.411,...,1.0,0.6,0.4,26580000,1,0,0,0,0,0


## Playoff 36 raw

In [8]:
import os
import pandas as pd
import unicodedata


def normalize_name(value):
    text = unicodedata.normalize('NFKD', str(value))
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text.strip().lower()


raw_path = 'data/playoff_36_raw.csv' if os.path.exists('data/playoff_36_raw.csv') else 'playoff_36_raw.csv'
adv_salary_path = 'data/playoff_adv_salary.csv' if os.path.exists('data/playoff_adv_salary.csv') else 'playoff_adv_salary.csv'

raw_df = pd.read_csv(raw_path, encoding='latin1')
adv_salary_df = pd.read_csv(adv_salary_path, encoding='latin1')

keep_cols = ['Player', 'Salary', 'Awards', 'Pos_C', 'Pos_PF', 'Pos_PG', 'Pos_SF', 'Pos_SG']
missing = [c for c in keep_cols if c not in adv_salary_df.columns]
if missing:
    raise ValueError(f'Missing expected columns in playoff_adv_salary.csv: {missing}')

lookup_df = adv_salary_df[keep_cols].drop_duplicates(subset=['Player']).copy()
lookup_df['name_key'] = lookup_df['Player'].map(normalize_name)
raw_df['name_key'] = raw_df['Player'].map(normalize_name)

# Keep only playoff_36_raw players that exist in playoff_adv_salary.csv
lookup_names = set(lookup_df['name_key'])
raw_df = raw_df[raw_df['name_key'].isin(lookup_names)].copy()

merged_df = raw_df.merge(
    lookup_df.drop(columns=['Player']),
    on='name_key',
    how='left'
).drop(columns=['name_key'])

output_path = 'data/playoff_36_salary.csv'
merged_df.to_csv(output_path, index=False)

matched_rows = merged_df['Salary'].notna().sum()
print(f'Loaded raw file: {raw_path}')
print(f'Loaded salary file: {adv_salary_path}')
print(f'Rows kept after player match filter: {len(merged_df)}')
print(f'Matched Salary/Awards/Pos rows: {matched_rows}/{len(merged_df)}')
print(f'Saved: {output_path}')
merged_df.head(20)


Loaded raw file: data/playoff_36_raw.csv
Loaded salary file: data/playoff_adv_salary.csv
Rows kept after player match filter: 116
Matched Salary/Awards/Pos rows: 116/116
Saved: data/playoff_36_salary.csv


,ï»¿Rk,Player,Age,Team,Pos,G,GS,MP,MPG,FG,...,PF,PTS,Unnamed: 31,Salary,Awards,Pos_C,Pos_PF,Pos_PG,Pos_SF,Pos_SG
0,1.0,Shai Gilgeous-Alexander,26.0,OKC,PG,23.0,23.0,851.0,37.000000,9.9,...,2.7,29.1,NaN,61005000,1,0,0,1,0,0
1,2.0,Jalen Williams,23.0,OKC,SG,23.0,23.0,796.0,34.608696,8.1,...,2.3,22.3,NaN,41500000,1,0,0,0,0,1
2,3.0,Tyrese Haliburton,24.0,IND,PG,23.0,23.0,772.0,33.565217,6.8,...,1.8,18.6,NaN,45550512,0,0,0,1,0,0
3,4.0,Pascal Siakam,30.0,IND,PF,23.0,23.0,771.0,33.521739,8.2,...,3.1,22.0,NaN,45550512,1,0,1,0,0,0
4,5.0,Andrew Nembhard,25.0,IND,SG,23.0,23.0,769.0,33.434783,4.9,...,2.9,13.4,NaN,18797619,0,0,0,0,0,1
5,6.0,Mikal Bridges,28.0,NYK,SF,18.0,18.0,706.0,39.222222,6.1,...,2.0,14.3,NaN,33482145,0,0,0,0,1,0
6,7.0,OG Anunoby,27.0,NYK,PF,18.0,18.0,705.0,39.166667,5.3,...,2.5,15.0,NaN,39568966,0,0,1,0,0,0
7,8.0,Chet Holmgren,22.0,OKC,C,23.0,23.0,686.0,29.826087,6.5,...,2.5,18.3,NaN,41500000,0,1,0,0,0,0
8,9.0,Jalen Brunson,28.0,NYK,PG,18.0,18.0,680.0,37.777778,9.6,...,3.1,28.1,NaN,35000000,1,0,0,1,0,0
9,10.0,Myles Turner,28.0,IND,C,23.0,23.0,675.0,29.347826,5.7,...,3.6,16.9,NaN,26580000,0,1,0,0,0,0
